# **Sesión: Uso de AI para análisis de datos**

**Objetivo:** Integrar AI a nuestro workflow de manera más nativa.

---

## **Estructura de la clase**

---

### **1. Breve explicación de API (15 minutos)**

**API:** Le permite a una aplicación pedir data o servicios a otra aplicación.

![Alt text](assets/api_request.png)

---

### **2. Llamada a la API de OpenAI (20 minutos)**

**Objetivo:** Hacer una primer llamada a la API de OpenAI y conocer la estructura de la respuesta.

**Conceptos importantes:**

- **Tokens**

- **Prompting**
    - Hecho mediante una lista de diccionarios con "roles" y "contenido". 
    - **system:** Define la personalidad y el tono de las interacciones. Sirve como instrucciones guía. 
    - **user:** La instrucción específica dada por el usuario. 
    - **assistant:** Respuesta del LLM. 

In [25]:
from openai import OpenAI 
from dotenv import load_dotenv
import os

load_dotenv()

True

In [26]:
client = OpenAI()

completion = client.chat.completions.create(
  model="gpt-4o",
  messages=[
    {"role": "system", "content": "Your name is Ginger, you are a Data Scientist."},
    {"role": "user", "content": "Hello! What is your name and profession?"}
  ]
)

Todo el objeto que recibimos de respuesta:

In [27]:
print(completion)

ChatCompletion(id='chatcmpl-AtmShDLo76KooGGa3I7EkVyOhjB8t', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! My name is Ginger, and I am a Data Scientist.', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None))], created=1737857691, model='gpt-4o-2024-08-06', object='chat.completion', service_tier='default', system_fingerprint='fp_50cad350e4', usage=CompletionUsage(completion_tokens=15, prompt_tokens=31, total_tokens=46, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


¿Cuántos tokens usamos?

In [28]:
print(completion.usage.total_tokens)

46


Puedes obtener más de una respuesta (_choices_); hoy sólo obtendremos una

In [29]:
print(completion.choices)

[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! My name is Ginger, and I am a Data Scientist.', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None))]


In [30]:
print(completion.choices[0].message.content)

Hello! My name is Ginger, and I am a Data Scientist.


---

### **4. Chat multiturno (30 minutos)**

**Objetivo:** Cómo hablar con Chat-GPT en una función multiturno.

Hacemos unas funciones para que nos ayuden a leer las respuestas. 

**NO SON NECESARIAS**, pero nos será más fácil leer. 

In [31]:
def write_to_markdown_file(snippet, message_sender="user", file_path="chat_output.md"):
    with open(file_path, "a", encoding="utf-8") as f:
        f.write(f"**{message_sender}:**" + "\n\n" + snippet + "\n\n")

def delete_file(file_path):
    if os.path.isfile(file_path):
        os.remove(file_path)
    else: 
        print('File does not exist')

In [32]:
delete_file("chat_output.md")

Necesitamos una función para acumular los mensajes. 

Iniciamos con el system prompt.

In [33]:
messages_for_llm = [{"role":"system","content":"Eres un analista de datos Senior con amplia experiencia en Python.\
             Provees información de manera amigable y fácil de entender."}]

Creamos una función donde el _prompt_ del usuario se formatee para añadirse a la lista de mensajes. 

También se añade la respuesta del asistente. 

**NOTA:** Hay mucho mejores maneras de hacer esta función, hoy la dejaremos así. 

In [34]:
def multiturn_conversation(user_prompt, message_history = messages_for_llm):

    format_for_messages = {"role":"user","content":user_prompt}

    write_to_markdown_file(user_prompt, message_sender="user")

    message_history.append(format_for_messages)

    completions = client.chat.completions.create(
        model="gpt-4o",
        messages=message_history
    )

    assistant_response = completions.choices[0].message.content

    format_for_messages = {"role":"assistant","content":assistant_response}

    message_history.append(format_for_messages)

    write_to_markdown_file(assistant_response, message_sender="assistant")

    return print(assistant_response)

In [35]:
multiturn_conversation("Hola! En qué trabajas?")

¡Hola! Trabajo como analista de datos, principalmente utilizando Python para explorar, analizar y visualizar conjuntos de datos. Mi objetivo es convertir datos complejos en información clara y comprensible para ayudar a tomar decisiones informadas. Si tienes alguna pregunta sobre datos o necesitas ayuda con Python, ¡estaré encantado de ayudarte!


In [36]:
multiturn_conversation("Si te paso un CSV, me puedes ayudar")

¡Por supuesto! Estaré encantado de ayudarte con cualquier CSV que tengas. Puedes empezar por describirme el contenido del archivo o los detalles de lo que necesitas. Si tienes preguntas sobre alguna tarea específica que quieras realizar con el CSV, como limpiar los datos, analizar una tendencia o crear visualizaciones, dime y te guiaré en el proceso.


Todos los mensajes los toma.

In [37]:
print(messages_for_llm)

[{'role': 'system', 'content': 'Eres un analista de datos Senior con amplia experiencia en Python.             Provees información de manera amigable y fácil de entender.'}, {'role': 'user', 'content': 'Hola! En qué trabajas?'}, {'role': 'assistant', 'content': '¡Hola! Trabajo como analista de datos, principalmente utilizando Python para explorar, analizar y visualizar conjuntos de datos. Mi objetivo es convertir datos complejos en información clara y comprensible para ayudar a tomar decisiones informadas. Si tienes alguna pregunta sobre datos o necesitas ayuda con Python, ¡estaré encantado de ayudarte!'}, {'role': 'user', 'content': 'Si te paso un CSV, me puedes ayudar'}, {'role': 'assistant', 'content': '¡Por supuesto! Estaré encantado de ayudarte con cualquier CSV que tengas. Puedes empezar por describirme el contenido del archivo o los detalles de lo que necesitas. Si tienes preguntas sobre alguna tarea específica que quieras realizar con el CSV, como limpiar los datos, analizar un

---

### **4. Caso práctico (35 minutos)**

**Escenario:**
Tienes 10 minutos para preparar un reporte para el gerente del Banco Mundial que conteste las siguientes preguntas. Tu Excel no sirve.  

1. ¿Cuál fue la media de crecimiento de GDP de México de 2010 a 2015?
2. ¿Cómo se compara esta media de crecimiento con la de Estados Unidos en el mismo período?
3. ¿Cuál es el país con la menor media de crecimiento de 2010 a 2015?

**Instrucciones:**

- Utiliza la conversación con Chat-GPT dentro del Notebook para ayudarte. 

In [38]:
import pandas as pd

PATH_TO_FILE = 'data/world indicators-gdp_growth.csv'

df = pd.read_csv(PATH_TO_FILE)

In [39]:
df.head(10)

,indicator,country,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015
0,GDP growth (annual %),Argentina,8.047152,9.007651,4.057233,-5.918525,10.125398,6.003952,-1.026420,2.405324,-2.512615,2.731160
1,GDP growth (annual %),Brazil,3.961989,6.069871,5.094195,-0.125812,7.528226,3.974423,1.921176,3.004823,0.503956,-3.545763
2,GDP growth (annual %),Canada,2.637944,2.049905,0.995406,-2.915086,3.090806,3.137194,1.755661,2.325814,2.873467,0.649971
3,GDP growth (annual %),Chile,6.049991,5.168231,3.789393,-1.118037,5.851651,6.223897,6.155340,3.308508,1.792649,2.151942
4,GDP growth (annual %),Colombia,6.716869,6.738195,3.283446,1.139649,4.494659,6.947892,3.912636,5.133994,4.499030,2.955901
5,GDP growth (annual %),Cuba,12.065863,7.262137,4.116828,1.451305,2.390352,2.802301,3.014900,2.747603,1.047577,4.438334
6,GDP growth (annual %),Honduras,6.567244,6.188327,4.231600,-2.431628,3.731140,3.835691,4.128688,2.791560,3.058081,3.840080
7,GDP growth (annual %),Mexico,4.805014,2.077864,0.943332,-6.295251,4.971335,3.444045,3.553211,0.852102,2.503764,2.702323
8,GDP growth (annual %),Peru,7.528899,8.518388,9.126568,1.095824,8.332459,6.327192,6.139725,5.852518,2.382157,3.252245
9,GDP growth (annual %),United States,2.784540,2.003858,0.113587,-2.576500,2.695193,1.564407,2.289113,2.117830,2.523820,2.945550


Borramos nuestra antigua conversación.

No es necesario. 

In [48]:
delete_file("chat_output.md")

**System Prompt**

- ¿Quién queremos que sea esta vez?

In [49]:
messages_for_llm = [{"role":"system","content":"Eres un analista de datos financiero Senior con amplia experiencia en Python.\
             Provees información de manera amigable y fácil de entender."}]

Le pasamos un **pequeño sample**

In [50]:
multiturn_conversation("""Tengo un dataframe en Pandas que se ve así: 
indicator	country	2006	2007	2008	2009	2010	2011	2012	2013	2014	2015
0	GDP growth (annual %)	Argentina	8.047152	9.007651	4.057233	-5.918525	10.125398	6.003952	-1.026420	2.405324	-2.512615	2.731160
1	GDP growth (annual %)	Brazil	3.961989	6.069871	5.094195	-0.125812	7.528226	3.974423	1.921176	3.004823	0.503956	-3.545763

Este es un sample. El dataframe total tiene 10 filas. No es necesario que hagas nada por ahora.""")

¡Entendido! Ya sabes que tienes un DataFrame con información del crecimiento anual del PIB en porcentaje para diferentes países. Si en algún momento quieres realizar análisis o necesitas ayuda para manipular este DataFrame, como calcular medias, hacer comparaciones entre países o realizar visualizaciones, estaré encantado de asistirte. Si tienes otras preguntas, no dudes en preguntar. 😊


In [51]:
multiturn_conversation("""Responde las siguientes preguntas: 
                       
                        -¿Cuál fue la media de crecimiento de GDP de México de 2010 a 2015?
                       
                        -¿Cómo se compara esta media de crecimiento con la de Estados Unidos en el mismo período?
                       
                        -¿Cuál es el país con la menor media de crecimiento de 2010 a 2015?
                       
                       Sólo necesito el código de Python para responder estas preguntas.""")

¡Por supuesto! A continuación te proporciono el código en Python que puedes usar para responder estas preguntas utilizando Pandas:

```python
import pandas as pd

# Supongamos que tienes tu DataFrame en la variable llamada df

# Calculamos la media de crecimiento del PIB para México de 2010 a 2015
mexico_mean_growth = df.loc[df['country'] == 'Mexico', '2010':'2015'].mean(axis=1).values[0]

# Calculamos la media de crecimiento del PIB para Estados Unidos de 2010 a 2015
us_mean_growth = df.loc[df['country'] == 'United States', '2010':'2015'].mean(axis=1).values[0]

# Comparación entre la media de México y Estados Unidos
comparison = mexico_mean_growth - us_mean_growth

# Calculamos la media de crecimiento del PIB para todos los países de 2010 a 2015
mean_growth_all_countries = df.loc[:, '2010':'2015'].mean(axis=1)

# Encontramos el país con la menor media de crecimiento en ese período
country_with_min_growth = df.loc[mean_growth_all_countries.idxmin(), 'country']

print(f"La media de cre

In [52]:
# Calculamos la media de crecimiento del GDP para México de 2010 a 2015
mexico_mean_growth = df.loc[df['country'] == 'Mexico', '2010':'2015'].mean(axis=1).values[0]

# Calculamos la media de crecimiento del GDP para Estados Unidos de 2010 a 2015
us_mean_growth = df.loc[df['country'] == 'United States', '2010':'2015'].mean(axis=1).values[0]

# Comparación entre la media de México y Estados Unidos
comparison = mexico_mean_growth - us_mean_growth

# Calculamos la media de crecimiento del GDP para todos los países de 2010 a 2015
mean_growth_all_countries = df.loc[:, '2010':'2015'].mean(axis=1)

# Encontramos el país con la menor media de crecimiento en ese período
country_with_min_growth = df.loc[mean_growth_all_countries.idxmin(), 'country']

print(f"La media de crecimiento de GDP de México de 2010 a 2015 es: {mexico_mean_growth}")
print(f"La media de crecimiento de GDP de Estados Unidos de 2010 a 2015 es: {us_mean_growth}")
print(f"La diferencia entre México y Estados Unidos es: {comparison}")
print(f"El país con la menor media de crecimiento de 2010 a 2015 es: {country_with_min_growth}")

La media de crecimiento de GDP de México de 2010 a 2015 es: 3.004463148233333
La media de crecimiento de GDP de Estados Unidos de 2010 a 2015 es: 2.3559855323333334
La diferencia entre México y Estados Unidos es: 0.6484776158999996
El país con la menor media de crecimiento de 2010 a 2015 es: Brazil


**Comprobemos los resultados**

In [53]:
df[df['country'] == 'Mexico'].loc[:, '2010':'2015'].mean(axis=1)

7    3.004463
dtype: float64

In [54]:
df[df['country'] == 'United States'].loc[:, '2010':'2015'].mean(axis=1)

9    2.355986
dtype: float64

In [55]:
indice_del_menor = df.loc[:, '2010':'2015'].mean(axis=1).idxmin()
df.loc[indice_del_menor, "country"]

'Brazil'

---

### **Reporte de Laboratorio**

**Ejercicio esperado:**

Entregar, por equipos de 3, los ejercicios del inciso 4. 

Usen el mismo dataset 'world indicators-gdp_growth.csv'

**Conclusiones:**

Analicen y respondan lo siguiente: 

- ¿Cómo es usar Chat-GPT distinto a usarlo mediante API?

- ¿Qué le hace falta a la API para ser una _experiencia completa_?

In [89]:
delete_file("chat_output.md")